**특정 URL에서 원하는 이미지 크롤링하기**

In [ ]:
import os
import json
import requests
from bs4 import BeautifulSoup
import shutil
from google.colab import files
from urllib.parse import urlparse

def scrape_archdaily_project(url, project_id="arch_ref_001", base_folder="dataset"):
    # 1. 프로젝트 폴더 생성 (이미지와 JSON을 한 세트로 묶음)
    project_folder = os.path.join(base_folder, project_id)
    os.makedirs(project_folder, exist_ok=True)
    print(f"📁 프로젝트 폴더 생성: {project_folder}")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        print(f"🌐 데이터 분석 중: {url}")
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # ---------------------------------------------------
        # [Step 1] 텍스트 및 메타데이터 크롤링
        # ---------------------------------------------------

        # 1-1. 본문 텍스트 (Description) 추출
        description_divs = soup.find_all('div', class_='post-content')
        text_description = ""
        if description_divs:
            paragraphs = description_divs[0].find_all('p')
            text_description = " ".join([p.get_text(strip=True) for p in paragraphs])

        # 1-2. 건축 스펙 (Area 등 정량 데이터) 추출
        specs_dict = {}
        specs_ul = soup.find('ul', class_='afd-specs__list')
        if specs_ul:
            for li in specs_ul.find_all('li'):
                title = li.find('h3')
                value = li.find('div', class_='afd-specs__value') or li.find('a')
                if title and value:
                    specs_dict[title.get_text(strip=True)] = value.get_text(strip=True)

        # 면적(Area) 숫자로 변환 처리 시도
        area_sqm = 0
        if 'Area' in specs_dict:
            # 예: "120 m²" -> 120
            import re
            numbers = re.findall(r'\d+', specs_dict['Area'])
            if numbers:
                area_sqm = int(numbers[0])

        # ---------------------------------------------------
        # [Step 2] 요청하신 JSON 스키마 구조화
        # ---------------------------------------------------
        # *주의: 완벽한 JSON 추출을 위해서는 추후 이 단계에서
        # text_description을 GPT-4o에 넣어 Function Calling으로 파싱해야 합니다.
        # 여기서는 크롤링 단계의 MVP 버전을 위해 기본 룰 기반 맵핑을 적용합니다.

        project_json = {
            "project_id": project_id,
            "metadata": {
                "source": "archdaily",
                "building_type": "residential" if "House" in soup.title.text else "commercial", # 임시 분류
                "gross_floor_area_sqm": area_sqm,
                "floors": 2, # 기본값 (본문 분석 필요)
                "layout_typology": "unknown", # LLM 파싱 필요 영역
                "core_spaces": ["living_room", "kitchen", "bedroom", "bathroom"], # 기본 세트
                "architectural_style": [], # LLM 파싱 필요 영역
                "materials": [], # LLM 파싱 필요 영역
                "atmosphere": [] # LLM 파싱 필요 영역
            },
            "text_description": text_description[:500] + "..." if len(text_description) > 500 else text_description # 너무 길면 자름
        }

        # 텍스트에 특정 키워드가 있으면 임시로 태깅 (MVP 테스트용)
        desc_lower = text_description.lower()
        if "concrete" in desc_lower: project_json["metadata"]["materials"].append("exposed_concrete")
        if "wood" in desc_lower or "timber" in desc_lower: project_json["metadata"]["materials"].append("warm_wood")
        if "minimal" in desc_lower: project_json["metadata"]["architectural_style"].append("minimalist")
        if "light" in desc_lower: project_json["metadata"]["atmosphere"].append("natural_light")
        if "courtyard" in desc_lower: project_json["metadata"]["layout_typology"] = "courtyard"

        # JSON 파일 저장
        json_filename = os.path.join(project_folder, f"{project_id}_metadata.json")
        with open(json_filename, 'w', encoding='utf-8') as f:
            json.dump(project_json, f, ensure_ascii=False, indent=2)
        print("📄 JSON 메타데이터 생성 완료")

        # ---------------------------------------------------
        # [Step 3] 고해상도 이미지 크롤링 (이전 해결책 적용)
        # ---------------------------------------------------
        images = soup.find_all(['img', 'source'])
        downloaded_count = 0
        seen_urls = set()

        for tag in images:
            raw_url = tag.get('data-src') or tag.get('src') or tag.get('srcset')
            if not raw_url: continue

            img_url = raw_url.split(' ')[0]

            if 'adsttc.com' in img_url and img_url not in seen_urls:
                try:
                    img_data = requests.get(img_url, headers=headers).content

                    if len(img_data) < 50000: # 50KB 이하 아이콘/썸네일 제외
                        continue

                    parsed_url = urlparse(img_url)
                    ext = os.path.splitext(parsed_url.path)[1].lower()
                    if not ext: ext = ".jpg"

                    # 이미지 파일 저장
                    img_filename = os.path.join(project_folder, f"img_{downloaded_count:03d}{ext}")
                    with open(img_filename, 'wb') as f:
                        f.write(img_data)

                    seen_urls.add(img_url)
                    downloaded_count += 1
                except Exception as e:
                    pass

        print(f"🖼️ 총 {downloaded_count}개의 고해상도 이미지 다운로드 완료!")
        print(f"✅ [{project_id}] 세트 구축 완료!\n")

    except Exception as e:
        print(f"❌ 크롤링 오류: {e}")

# ==========================================
# 🚀 실행 설정 부분
# ==========================================
base_dataset_folder = "cad_agent_dataset"

# 타겟 프로젝트 리스트 (예시로 2개 세팅)
target_projects = [
    {
        "id": "arch_ref_001",
        "url": "https://www.archdaily.com/1015112/house-in-kashiwa-yoshimura-yasutaka-architects"
    },
    {
        "id": "arch_ref_002",
        "url": "https://www.archdaily.com/1015096/v-house-h-plus-f-architects"
    }
]

# 1. 크롤링 실행 (리스트를 돌며 폴더별로 저장)
for proj in target_projects:
    scrape_archdaily_project(proj["url"], proj["id"], base_dataset_folder)

# 2. 코랩 로컬에 저장된 데이터셋 전체를 ZIP 파일로 압축
zip_filename = f"{base_dataset_folder}.zip"
shutil.make_archive(base_dataset_folder, 'zip', base_dataset_folder)
print(f"\n📦 '{zip_filename}' 파일로 압축 중...")

# 3. 내 PC로 자동 다운로드
files.download(zip_filename)

📁 프로젝트 폴더 생성: cad_agent_dataset/arch_ref_001
🌐 데이터 분석 중: https://www.archdaily.com/1015112/house-in-kashiwa-yoshimura-yasutaka-architects
📄 JSON 메타데이터 생성 완료
🖼️ 총 11개의 고해상도 이미지 다운로드 완료!
✅ [arch_ref_001] 세트 구축 완료!

📁 프로젝트 폴더 생성: cad_agent_dataset/arch_ref_002
🌐 데이터 분석 중: https://www.archdaily.com/1015096/v-house-h-plus-f-architects
📄 JSON 메타데이터 생성 완료
🖼️ 총 6개의 고해상도 이미지 다운로드 완료!
✅ [arch_ref_002] 세트 구축 완료!


📦 'cad_agent_dataset.zip' 파일로 압축 중...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>